<a href="https://colab.research.google.com/github/Alessandro-json/MimoAI/blob/main/Cloud_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Colab for CLOUD COMPUTING

###pip install

In [ ]:
!pip install torch torchvision av pandas
!pip -q install -U yt-dlp
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install av scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 130.2 MB/s  0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


##imports

In [ ]:
import os
import time
import av
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from torchvision.models.video import r3d_18, R3D_18_Weights
import av
import time
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from torchvision.models.video import (
    r3d_18,
    R3D_18_Weights
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

#importa MIMETICS

import the dataset containing mimed actions. I'm going to finetune a pretrained model on kinetics

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Mimetics_release_v1.0.zip to Mimetics_release_v1.0.zip


unzip mimetics

In [ ]:
!unzip -q Mimetics_release_v1.0.zip -d mimetics_dataset

Let's see the columns

In [ ]:
import pandas as pd

csv_path = "mimetics_dataset/mimetics_v1.0.csv"

df = pd.read_csv(csv_path)

print(df.columns.tolist())
display(df.head(10))
print("Numero righe:", len(df))

['label', 'youtube_id', 'time_start', 'time_end', 'split', 'is_cc']


,label,youtube_id,time_start,time_end,split,is_cc
0,archery,0zFbEjoGlhc,6,10,test,0
1,archery,0zFbEjoGlhc,14,24,test,0
2,archery,AHMNZopmQBQ,71,74,test,0
3,archery,GZWM9or9AhM,40,44,test,0
4,archery,N4WKMa_KKYc,23,24,test,0
5,archery,Nru0Tb3f6lg,14,15,test,0
6,archery,SHVMWGGAkGQ,104,109,test,0
7,archery,SdY0GFiME2c,393,395,test,0
8,archery,Wj4-fNnZqIc,3,7,test,0
9,archery,aXknIbHLPLM,13,16,test,0


Numero righe: 713


#DOwnload

In [ ]:
import pandas as pd
import subprocess
import re
from pathlib import Path

# =========================================
# CONFIG
# =========================================

CSV_PATH = "mimetics_dataset/mimetics_v1.0.csv"
OUTPUT_DIR = Path("mimetics_videos")
OUTPUT_DIR.mkdir(exist_ok=True)

#they have to be downloaded from youtube
MAX_NEW_DOWNLOADS = None

# =========================================
# LOAD CSV
# =========================================

df = pd.read_csv(CSV_PATH)

print("Numero totale clip nel CSV:", len(df))
display(df.head())


# =========================================
# HELPER
# =========================================

def safe_name(text):
    text = str(text).lower().strip()
    text = text.replace(" ", "_")
    text = re.sub(r"[^\w\-]", "", text)
    return text


# =========================================
# DOWNLOAD LOOP
# =========================================

downloaded_now = 0
skipped_existing = 0
failed = 0

success_rows = []
failed_rows = []
skipped_rows = []

for idx, row in df.iterrows():

    label = str(row["label"])
    youtube_id = str(row["youtube_id"])
    start = float(row["time_start"])
    end = float(row["time_end"])

    class_name = safe_name(label)
    class_dir = OUTPUT_DIR / class_name
    class_dir.mkdir(parents=True, exist_ok=True)

    # Nome base del file
    file_prefix = f"{idx:04d}_{class_name}"

    # Se esiste già QUALSIASI file con quel prefisso, salta
    existing_files = list(class_dir.glob(f"{file_prefix}.*"))
    if len(existing_files) > 0:
        skipped_existing += 1
        skipped_rows.append({
            "idx": idx,
            "label": label,
            "youtube_id": youtube_id,
            "status": "already_exists",
            "path": str(existing_files[0])
        })
        continue

    # Se vuoi limitare il numero di NUOVI download
    if MAX_NEW_DOWNLOADS is not None and downloaded_now >= MAX_NEW_DOWNLOADS:
        break

    url = f"https://www.youtube.com/watch?v={youtube_id}"
    output_template = str(class_dir / f"{file_prefix}.%(ext)s")

    print("\n------------------------------------")
    print(f"[CSV row {idx}]")
    print("Classe:", label)
    print("YouTube:", youtube_id)
    print(f"Intervallo: {start:.2f}s → {end:.2f}s")

    cmd = [
        "yt-dlp",
        "--no-playlist",
        "--download-sections", f"*{start}-{end}",
        "--force-keyframes-at-cuts",
        "-f", "bv*[height<=480]+ba/b[height<=480]",
        "--merge-output-format", "mp4",
        "-o", output_template,
        url
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        downloaded_now += 1
        success_rows.append({
            "idx": idx,
            "label": label,
            "youtube_id": youtube_id,
            "time_start": start,
            "time_end": end,
            "status": "downloaded"
        })
        print("✓ Scaricato")
    else:
        failed += 1
        failed_rows.append({
            "idx": idx,
            "label": label,
            "youtube_id": youtube_id,
            "time_start": start,
            "time_end": end,
            "status": "failed",
            "error_tail": result.stderr[-500:]
        })
        print("✗ Fallito")
        print(result.stderr[-500:])


print("\n====================================")
print("DOWNLOAD TERMINATO")
print("====================================")
print("Nuovi scaricati:", downloaded_now)
print("Già presenti / saltati:", skipped_existing)
print("Falliti:", failed)

Numero totale clip nel CSV: 713


,label,youtube_id,time_start,time_end,split,is_cc
0,archery,0zFbEjoGlhc,6,10,test,0
1,archery,0zFbEjoGlhc,14,24,test,0
2,archery,AHMNZopmQBQ,71,74,test,0
3,archery,GZWM9or9AhM,40,44,test,0
4,archery,N4WKMa_KKYc,23,24,test,0


Output streaming troncato alle ultime 5000 righe.
YouTube: TBdnqWOtY-o
Intervallo: 0.00s → 8.00s
✗ Fallito
ome formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one
ERROR: [youtube] TBdnqWOtY-o: Please sign in. This helps protect our community. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


------------------------------------
[CSV row 61]
Classe: brushing teeth
YouTube: _HDQfm5f36U
Intervallo: 37.00s → 42.00s
✓ Scaricato

------------------------------------
[CSV row 62]
Classe: brushing teeth
YouTube: a5SMXR1BUL0
Intervallo: 10.00s → 20.00s
✗ Fallito
tube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runt

Logs of download to understand the failed one

In [ ]:
success_df = pd.DataFrame(success_rows)
failed_df = pd.DataFrame(failed_rows)
skipped_df = pd.DataFrame(skipped_rows)

success_df.to_csv("mimetics_downloaded_now.csv", index=False)
failed_df.to_csv("mimetics_failed.csv", index=False)
skipped_df.to_csv("mimetics_skipped_existing.csv", index=False)

print("Log salvati.")

Log salvati.


In [ ]:
from pathlib import Path
import pandas as pd

video_files = list(Path("mimetics_videos").rglob("*.mp4"))

print("Totale video .mp4 trovati:", len(video_files))

class_counts = {}

for video in video_files:
    class_name = video.parent.name
    class_counts[class_name] = class_counts.get(class_name, 0) + 1

class_df = pd.DataFrame(
    sorted(class_counts.items(), key=lambda x: x[0]),
    columns=["class", "num_videos"]
)

display(class_df)
print("Numero classi scaricate:", len(class_df))

Totale video .mp4 trovati: 567


,class,num_videos
0,archery,16
1,bowling,10
2,brushing_hair,17
3,brushing_teeth,8
4,canoeing_or_kayaking,12
5,catching_or_throwing_baseball,12
6,catching_or_throwing_frisbee,9
7,clean_and_jerk,10
8,cleaning_windows,14
9,climbing_a_rope,13


Numero classi scaricate: 50


#DRIVE MOUNT


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Save the videos on private drive

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/mimetics_videos")
dst = Path("/content/drive/MyDrive/MimoAI/datasets/mimetics_videos")

dst.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    src,
    dst,
    dirs_exist_ok=True
)

print("Dataset salvato su Drive")

FileNotFoundError: [Errno 2] No such file or directory: '/content/mimetics_videos'

In [ ]:
#DATASET_DIR = "/content/drive/MyDrive/MimoAI/datasets/mimetics_videos"
DATASET_DIR = Path(
    "/content/drive/MyDrive/MimoAI/datasets/mimetics_videos"
)